## 🎮 Data Abstraction Quest — Gamified Review

Test your knowledge in an interactive RPG! Explore the world below and talk to 5 NPC teachers — each one quizzes you on a different concept from this lesson.

| Key | Action |
|-----|--------|
| **WASD** | Move your character |
| **E** | Talk to an NPC (walk into them first) |

| NPC | Concept | Topic |
|-----|---------|-------|
| List Keeper (wizard) | Lists | Ordered sequences of elements |
| Index Oracle (robot) | Indexing | 0-based access, string indexing |
| Loop Sage (octocat) | Loops + Lists | Computing with iteration |
| Abstraction Architect (penguin) | Why Abstraction? | Managing complexity |
| AP Guardian (nerd) | AP Exam | 1-based indexing, pseudocode |

Complete all 5 to become **Master of Abstraction!** Watch your score in the top-right HUD.

In [ ]:
%%js

// GAME_RUNNER: Select your persona from the in-game panel, then explore!

import GameControl from '/assets/js/GameEnginev1/essentials/GameControl.js';
import GameEnvBackground from '/assets/js/GameEnginev1/essentials/GameEnvBackground.js';
import Player from '/assets/js/GameEnginev1/essentials/Player.js';
import Npc from '/assets/js/GameEnginev1/essentials/Npc.js';

// ═══════════════════════════════════════════════════════════
//  Persona definitions
// ═══════════════════════════════════════════════════════════
var PERSONAS = [
  {
    id: 'Technologist',
    emoji: '💻',
    description: 'Driven solo coder. Tackles the hardest challenges.'
  },
  {
    id: 'Scrummer',
    emoji: '🤝',
    description: 'Team-first learner. Thrives in Agile collaboration.'
  },
  {
    id: 'Planner',
    emoji: '📋',
    description: 'Organized and process-driven. Champions the roadmap.'
  },
  {
    id: 'Closer',
    emoji: '✅',
    description: 'Detail-oriented finisher. Ships on time, every time.'
  }
];

// ═══════════════════════════════════════════════════════════
//  PersonaPanel — small in-game floating card (not fullscreen)
// ═══════════════════════════════════════════════════════════
var PersonaPanel = {
  el: null,

  show: function(onSelect) {
    if (this.el) return;

    this.el = document.createElement('div');
    this.el.style.cssText = [
      'position:fixed',
      'bottom:24px',
      'left:50%',
      'transform:translateX(-50%)',
      'z-index:9999',
      'background:rgba(22,22,46,0.93)',
      'border:2px solid #6c63ff',
      'border-radius:16px',
      'padding:16px 20px',
      'color:#fff',
      'font-family:Segoe UI,sans-serif',
      'box-shadow:0 0 24px rgba(100,100,255,0.35)',
      'display:flex',
      'flex-direction:column',
      'align-items:center',
      'gap:10px',
      'max-width:520px',
      'width:90%'
    ].join(';');

    var title = document.createElement('div');
    title.textContent = '🎭 Choose your persona';
    title.style.cssText = 'font-weight:bold;font-size:1em;color:#a8a4ff;letter-spacing:0.04em;';
    this.el.appendChild(title);

    var btnRow = document.createElement('div');
    btnRow.style.cssText = 'display:flex;gap:8px;flex-wrap:wrap;justify-content:center;';

    var self = this;
    PERSONAS.forEach(function(persona) {
      var btn = document.createElement('button');
      btn.style.cssText = [
        'padding:8px 14px',
        'border:2px solid #6c63ff',
        'border-radius:10px',
        'background:transparent',
        'color:#fff',
        'font-size:0.85em',
        'cursor:pointer',
        'transition:all 0.18s ease',
        'display:flex',
        'flex-direction:column',
        'align-items:center',
        'gap:2px',
        'min-width:100px'
      ].join(';');

      var emojiEl = document.createElement('span');
      emojiEl.textContent = persona.emoji;
      emojiEl.style.cssText = 'font-size:1.5em;';

      var nameEl = document.createElement('span');
      nameEl.textContent = persona.id;
      nameEl.style.cssText = 'font-weight:bold;font-size:0.9em;color:#a8a4ff;';

      btn.appendChild(emojiEl);
      btn.appendChild(nameEl);

      btn.onmouseenter = function() {
        btn.style.background = '#6c63ff44';
        btn.style.borderColor = '#a8a4ff';
      };
      btn.onmouseleave = function() {
        btn.style.background = 'transparent';
        btn.style.borderColor = '#6c63ff';
      };
      btn.onclick = function() {
        self.dismiss();
        onSelect(persona);
      };

      btnRow.appendChild(btn);
    });

    this.el.appendChild(btnRow);
    document.body.appendChild(this.el);
  },

  dismiss: function() {
    if (this.el) { this.el.remove(); this.el = null; }
  }
};

// ═══════════════════════════════════════════════════════════
//  PersonaLabel — pill that hovers above the player
// ═══════════════════════════════════════════════════════════
var PersonaLabel = {
  el: null,

  create: function(persona) {
    if (this.el) this.el.remove();

    this.el = document.createElement('div');
    this.el.style.cssText = [
      'position:fixed',
      'z-index:9000',
      'pointer-events:none',
      'background:rgba(26,26,46,0.92)',
      'border:2px solid #6c63ff',
      'border-radius:20px',
      'padding:3px 12px',
      'color:#fff',
      'font-family:Segoe UI,sans-serif',
      'font-size:0.82em',
      'font-weight:bold',
      'white-space:nowrap',
      'box-shadow:0 0 10px rgba(100,100,255,0.4)',
      'transform:translateX(-50%)'
    ].join(';');
    this.el.textContent = persona.emoji + ' ' + persona.id;
    document.body.appendChild(this.el);
  },

  update: function(playerEl) {
    if (!this.el || !playerEl) return;
    var rect = playerEl.getBoundingClientRect();
    var centerX = rect.left + rect.width / 2;
    var topY = rect.top - 26;
    this.el.style.left = centerX + 'px';
    this.el.style.top = topY + 'px';
  },

  destroy: function() {
    if (this.el) { this.el.remove(); this.el = null; }
  }
};

// ═══════════════════════════════════════════════════════════
//  PersonaLevel
// ═══════════════════════════════════════════════════════════
class PersonaLevel {
  constructor(gameEnv) {
    var path = gameEnv.path;
    var width = gameEnv.innerWidth;
    var height = gameEnv.innerHeight;

    // Show small panel docked to bottom — game still fully visible behind it
    PersonaPanel.show(function(persona) {
      PersonaLabel.create(persona);

      // Track player canvas position every 30ms
      if (window._personaLabelInterval) clearInterval(window._personaLabelInterval);
      window._personaLabelInterval = setInterval(function() {
        var canvases = document.querySelectorAll('canvas');
        var playerCanvas = canvases[1] || canvases[0];
        if (playerCanvas) PersonaLabel.update(playerCanvas);
      }, 30);
    });

    var bgData = {
      name: 'classroom',
      greeting: 'Welcome! Choose your persona below.',
      src: path + '/images/gamify/city.png',
      pixels: { height: 654, width: 966 }
    };

    var SCALE = 5;
    var playerData = {
      id: 'Student',
      greeting: 'Ready to collaborate!',
      src: path + '/images/gamify/chillguy.png',
      SCALE_FACTOR: SCALE,
      STEP_FACTOR: 1000,
      ANIMATION_RATE: 50,
      INIT_POSITION: { x: width / 2 - 30, y: height - (height / SCALE) },
      pixels: { height: 384, width: 512 },
      orientation: { rows: 3, columns: 4 },
      down:      { row: 0, start: 0, columns: 3 },
      downRight: { row: 1, start: 0, columns: 3, rotate: Math.PI / 16 },
      downLeft:  { row: 2, start: 0, columns: 3, rotate: -Math.PI / 16 },
      left:      { row: 2, start: 0, columns: 3 },
      right:     { row: 1, start: 0, columns: 3 },
      up:        { row: 3, start: 0, columns: 3 },
      upLeft:    { row: 2, start: 0, columns: 3, rotate: Math.PI / 16 },
      upRight:   { row: 1, start: 0, columns: 3, rotate: -Math.PI / 16 },
      hitbox: { widthPercentage: 0.45, heightPercentage: 0.2 },
      keypress: { up: 87, left: 65, down: 83, right: 68 }
    };

    function npcData(id, greeting, sprite, px, initPos, scaleFactor, orient, downAnim) {
      return {
        id: id,
        greeting: greeting,
        src: path + sprite,
        SCALE_FACTOR: scaleFactor,
        ANIMATION_RATE: 100,
        pixels: px,
        INIT_POSITION: initPos,
        orientation: orient,
        down: downAnim,
        hitbox: { widthPercentage: 0.1, heightPercentage: 0.2 },
        reaction: function() { alert(id + ': ' + greeting); },
        interact: function() { alert(id + ': ' + greeting); }
      };
    }

    var npc1 = npcData(
      'Technologist', '💻 I tackle the hardest challenges solo!',
      '/images/gamify/wizard.png', { width: 163, height: 185 },
      { x: width * 0.20, y: height * 0.35 }, 3,
      { rows: 1, columns: 1 }, { row: 0, start: 0, columns: 1 }
    );

    var npc2 = npcData(
      'Scrummer', '🤝 Teamwork makes the dream work!',
      '/images/gamify/robot.png', { width: 627, height: 316 },
      { x: width * 0.40, y: height * 0.35 }, 8,
      { rows: 3, columns: 6 }, { row: 1, start: 0, columns: 6 }
    );

    var npc3 = npcData(
      'Planner', '📋 I keep the whole project organized!',
      '/images/gamify/tux.png', { width: 256, height: 352 },
      { x: width * 0.60, y: height * 0.35 }, 4,
      { rows: 1, columns: 1 }, { row: 0, start: 0, columns: 1 }
    );

    var npc4 = npcData(
      'Closer', '✅ I get things done and ship on time!',
      '/images/gamify/nerd.png', { width: 343, height: 503 },
      { x: width * 0.78, y: height * 0.35 }, 4,
      { rows: 1, columns: 1 }, { row: 0, start: 0, columns: 1 }
    );

    this.classes = [
      { class: GameEnvBackground, data: bgData },
      { class: Player, data: playerData },
      { class: Npc, data: npc1 },
      { class: Npc, data: npc2 },
      { class: Npc, data: npc3 },
      { class: Npc, data: npc4 }
    ];
  }
}

export var gameLevelClasses = [PersonaLevel];
export { GameControl };

## Learning Objectives

> 3.A Generalize data sources through variables.  
> 
> 3.B Use abstraction to manage complexity in a program.
> 
> 3.C Explain how abstraction manages complexity.

## Data Abstractions

Data abstraction means *treating collections of values as a single unit* (like a list, array, or string), so you don’t have to think about all the small details every time you use them. This manages complexity by letting us use a name (like `numbers`) instead of repeatedly referencing each individual element.

For example, in Python:

```py
# A list in Python
numbers = [10, 20, 30, 40]

# Access by index (0-based in Python)
print(numbers[0]) # 10
print(numbers[2]) # 30

# Strings work similarly
word = "hello"
print(word[1]) # 'e'
```

or in JavaScript:

```js
// An array in JavaScript
let numbers = [10, 20, 30, 40];

console.log(numbers[0]); // 10
console.log(numbers[2]); // 30

// Strings are also indexable
let word = "hello";
console.log(word[1]); // 'e'
```

## Using Abstractions to Manage Complexity

Instead of writing code for each item, we use loops and functions to work with lists.

For example, in Python, we could iterate over a list and find the average of its elements:

```py
scores = [85, 92, 78, 90]

# Sum with a loop
total = 0
for score in scores:
    total += score
print("Average:", total / len(scores)) # 86.25
```

and in JS:

```js
let scores = [85, 92, 78, 90];

// Using a loop
let total = 0;
for (let score of scores) {
    total += score;
}
console.log("Average:", total / scores.length);
```

## How Abstraction Manages Complexity

Instead of thinking about *every score individually*, you can use the abstraction "list of scores."
- With a name (`scores`), you manipulate the entire group.
- You don't need to worry about how the list is stored in memory.
- Functions and loops let you work with the *concept* of a collection, not every detail.

What we mean by the last part may be a bit more understandable in some examples of benefits.
- Easier maintenance (add/remove scores without rewriting logic over and over again).
- Reusable functions (average can work for any list of numbers).
- Clearer code (you read "average of scores" instead of `score1 + score2 + score3 + ...`).

## Nuances

Remember that Python and JS both use 0-based indexing, which means that the first element of a list/array/string is at index 0 (like we showed before).

## Quick Facts (From AP)

- A *list* is an ordered sequence of elements. For example, `[value1, value2, value3, ...]` describes a list where `value1` is the first element, `value2` is the second element, and `value3` is the third element, and so on.
- An *element* is an individual value in a list that is assigned a unique index.
- An *index* is a common method for referencing the elements ina  list or string using natural numbers.
- A *string* is an ordered sequence of characters.
- Data abstraction provides a separation between the abstract properties of a data type and the concrete details of its representation.
- Data abstractions manage complexity in programs by giving a collection of data a name without referencing the specific details of the  representation.
- Data abstractions can be created using lists.
- Developing a data abstraction to implement in a program can result in a program that is easier to develop and maintain.
- Data abstractions often contain different types of elements.
- The use of lists allows multiple related items to be treated as a single value. Lists are referred to by different names such as array, depending on the programming language.
- **For AP, lists are 1-indexed!**